In [2]:
import ROOT
import re

def change_histogram_names(root_file_path):
    """
    Change histogram names in a ROOT file according to specific patterns.
    
    Pattern 1: SL_4j_resolved_<model>_<class> -> SL_4j_resolved___<model>__<class>
    Pattern 2: SL_4j_resolved_<model>_<class>_sub -> SL_4j_resolved__<class>___<model>__<class>
    """
    
    # Open the ROOT file in UPDATE mode to modify it
    root_file = ROOT.TFile.Open(root_file_path, "UPDATE")
    
    if not root_file or root_file.IsZombie():
        print(f"Error: Could not open file {root_file_path}")
        return
    
    # Get list of all keys (histogram names) in the file
    keys = root_file.GetListOfKeys()
    
    # Store original names and new names
    rename_map = {}
    
    # Process each histogram
    for key in keys:
        old_name = key.GetName()
        new_name = transform_histogram_name(old_name)
        
        if new_name != old_name:
            rename_map[old_name] = new_name
            print(f"Will rename: {old_name} -> {new_name}")
    
    # Perform the renaming
    for old_name, new_name in rename_map.items():
        # Get the histogram object
        hist = root_file.Get(old_name)
        if hist:
            # Set the new name
            hist.SetName(new_name)
            hist.SetTitle(new_name)  # Also update title if desired
            
            # Write the histogram with the new name
            hist.Write(new_name, ROOT.TObject.kOverwrite)
            
            # Delete the old histogram from the file
            root_file.Delete(f"{old_name};*")
            
            print(f"Renamed: {old_name} -> {new_name}")
        else:
            print(f"Warning: Could not find histogram {old_name}")
    
    # Save changes and close file
    root_file.Write()
    root_file.Close()
    
    print(f"Successfully updated {len(rename_map)} histogram names in {root_file_path}")

def transform_histogram_name(name):
    """
    Transform histogram names according to the specified patterns.
    
    Pattern 1: SL_4j_resolved_<model>_<class> -> SL_4j_resolved___<model>__<class>
    Pattern 2: SL_4j_resolved_<model>_<class>_sub -> SL_4j_resolved__<class>___<model>__<class>
    """
    
    # Pattern for: SL_4j_resolved_<model>_<class>_sub
    pattern1 = r'^(SL_4j_resolved)_(.+?)_([^_]+)_sub$'
    match1 = re.match(pattern1, name)
    
    if match1:
        prefix, model, nn_class = match1.groups()
        # Transform to: SL_4j_resolved__<class>___<model>__<class>
        new_name = f"{prefix}__{nn_class}___{model}__{nn_class}"
        return new_name
    
    # Pattern for: SL_4j_resolved_<model>_<class>
    pattern2 = r'^(SL_4j_resolved)_(.+?)_([^_]+)$'
    match2 = re.match(pattern2, name)
    
    if match2:
        prefix, model, nn_class = match2.groups()
        # Transform to: SL_4j_resolved___<model>__<class>
        new_name = f"{prefix}___{model}__{nn_class}"
        return new_name
    
    # Return original name if no pattern matches
    return name

def preview_changes(root_file_path):
    """
    Preview what changes would be made without actually modifying the file.
    """
    root_file = ROOT.TFile.Open(root_file_path, "READ")
    
    if not root_file or root_file.IsZombie():
        print(f"Error: Could not open file {root_file_path}")
        return
    
    keys = root_file.GetListOfKeys()
    changes_found = False
    
    print("Preview of changes:")
    print("-" * 60)
    
    for key in keys:
        old_name = key.GetName()
        new_name = transform_histogram_name(old_name)
        
        if new_name != old_name:
            print(f"{old_name} -> {new_name}")
            changes_found = True
    
    if not changes_found:
        print("No histograms match the renaming patterns.")
    
    root_file.Close()

Welcome to JupyROOT 6.30/02


In [5]:
from pathlib import Path
from references import functions, constants

workdir = Path('/eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_Rep/0806_NNInf_even')
resultsdir = workdir / 'results'

files = functions.get_mc_files(resultsdir)

for file in files:
    change_histogram_names(str(file))

Will rename: SL_4j_resolved_multi_HH_ttbar_tW_both_HH -> SL_4j_resolved___multi_HH_ttbar_tW_both__HH
Will rename: SL_4j_resolved_multi_HH_ttbar_tW_both_HH_sub -> SL_4j_resolved__HH___multi_HH_ttbar_tW_both__HH
Will rename: SL_4j_resolved_multi_HH_ttbar_tW_both_TTbar -> SL_4j_resolved___multi_HH_ttbar_tW_both__TTbar
Will rename: SL_4j_resolved_multi_HH_ttbar_tW_both_TTbar_sub -> SL_4j_resolved__TTbar___multi_HH_ttbar_tW_both__TTbar
Will rename: SL_4j_resolved_multi_HH_ttbar_tW_both_tW -> SL_4j_resolved___multi_HH_ttbar_tW_both__tW
Will rename: SL_4j_resolved_multi_HH_ttbar_tW_both_tW_sub -> SL_4j_resolved__tW___multi_HH_ttbar_tW_both__tW
Will rename: SL_4j_resolved_multi_HH_ttbar_tW_lrs_HH -> SL_4j_resolved___multi_HH_ttbar_tW_lrs__HH
Will rename: SL_4j_resolved_multi_HH_ttbar_tW_lrs_HH_sub -> SL_4j_resolved__HH___multi_HH_ttbar_tW_lrs__HH
Will rename: SL_4j_resolved_multi_HH_ttbar_tW_lrs_TTbar -> SL_4j_resolved___multi_HH_ttbar_tW_lrs__TTbar
Will rename: SL_4j_resolved_multi_HH_ttbar_t

In [1]:
import ROOT
def rename_hists(file_path):
    f = ROOT.TFile.Open(file_path, "UPDATE")
    keys = f.GetListOfKeys()

    for key in keys:
        obj = key.ReadObj()
        old_name = obj.GetName()

        if old_name.startswith("SL_4j_resolved_") and not old_name.startswith("SL_4j_resolved___"):
            new_name = old_name.replace("SL_4j_resolved_", "SL_4j_resolved___", 1)
            print(f"Renaming {old_name} → {new_name}")
            obj.SetName(new_name)
            obj.Write(new_name, ROOT.TObject.kOverwrite)

    f.Close()

Welcome to JupyROOT 6.30/02


In [2]:
from pathlib import Path
from references import functions, constants

workdir = Path('/eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_Rep/0805_LLR_odd')
resultsdir = workdir / 'results'
files = functions.get_mc_files(resultsdir)
for file in files:
    rename_hists(str(file))


Renaming SL_4j_resolved_bjets_mbb_llr → SL_4j_resolved___bjets_mbb_llr
Renaming SL_4j_resolved_bjets_dPhi_llr → SL_4j_resolved___bjets_dPhi_llr
Renaming SL_4j_resolved_bjets_dEta_llr → SL_4j_resolved___bjets_dEta_llr
Renaming SL_4j_resolved_bjets_dR_llr → SL_4j_resolved___bjets_dR_llr
Renaming SL_4j_resolved_bjets_pt_bb_llr → SL_4j_resolved___bjets_pt_bb_llr
Renaming SL_4j_resolved_bjet0_pt_llr → SL_4j_resolved___bjet0_pt_llr
Renaming SL_4j_resolved_bjet1_pt_llr → SL_4j_resolved___bjet1_pt_llr
Renaming SL_4j_resolved_bjets_mean_pt_llr → SL_4j_resolved___bjets_mean_pt_llr
Renaming SL_4j_resolved_trijet_mInv_llr → SL_4j_resolved___trijet_mInv_llr
Renaming SL_4j_resolved_trijet_pt_llr → SL_4j_resolved___trijet_pt_llr
Renaming SL_4j_resolved_trijet_pt_rat_llr → SL_4j_resolved___trijet_pt_rat_llr
Renaming SL_4j_resolved_trijet_bijet_dR_llr → SL_4j_resolved___trijet_bijet_dR_llr
Renaming SL_4j_resolved_trijet_bijet_dPhi_llr → SL_4j_resolved___trijet_bijet_dPhi_llr
Renaming SL_4j_resolved_tri

In [4]:
import ROOT

PREFIX_OLD = "SL_4j_resolved_"
PREFIX_NEW = "SL_4j_resolved___"

def is_histogram(obj):
    # Works for TH1, TH2, TH3 (and subclasses)
    return obj.InheritsFrom("TH1")

def cleanup_dir(tdir):
    """Delete old histogram keys whose new-name counterpart exists in the same directory."""
    # Only proceed if tdir is a TDirectory/TFile
    if not hasattr(tdir, "GetListOfKeys"):
        return

    keys = [k for k in tdir.GetListOfKeys()]  # snapshot

    for key in keys:
        cls = key.GetClassName()

        # Recurse into subdirectories only
        if cls.startswith("TDirectory"):
            subdir = key.ReadObj()  # TDirectory
            cleanup_dir(subdir)
            continue

        # Read the object to test its type (histogram vs tree/other)
        obj = key.ReadObj()

        # Skip non-hist objects (e.g., TTrees)
        if not is_histogram(obj):
            continue

        name = key.GetName()

        # Only target single-underscore names (not already fixed)
        if name.startswith(PREFIX_OLD) and not name.startswith(PREFIX_NEW):
            new_name = name.replace(PREFIX_OLD, PREFIX_NEW, 1)

            # Only delete the old one if the new one already exists in this directory
            if tdir.Get(new_name):
                print(f"[DELETE] {tdir.GetPath()}/{name}  (new exists: {new_name})")
                tdir.Delete(f"{name};*")  # delete all cycles of the old key
            else:
                print(f"[SKIP ] {tdir.GetPath()}/{name}  (no {new_name} found)")

def cleanup_file(path):
    print(f"\nProcessing: {path}")
    f = ROOT.TFile.Open(path, "UPDATE")
    if not f or f.IsZombie():
        print(f"[ERROR] Could not open {path}")
        return
    cleanup_dir(f)
    f.Write("", ROOT.TObject.kOverwrite)
    f.Close()
    print(f"[DONE ] {path}")

for file in files:
    cleanup_file(str(file))


Processing: /eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_Rep/0805_LLR_odd/results/TTbar_dl_2022.root
[DONE ] /eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_Rep/0805_LLR_odd/results/TTbar_dl_2022.root

Processing: /eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_Rep/0805_LLR_odd/results/TTbar_sl_2022.root
[DELETE] root://eoshome-a.cern.ch//eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_Rep/0805_LLR_odd/results/TTbar_sl_2022.root://SL_4j_resolved_bjets_mbb_llr  (new exists: SL_4j_resolved___bjets_mbb_llr)
[DELETE] root://eoshome-a.cern.ch//eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_Rep/0805_LLR_odd/results/TTbar_sl_2022.root://SL_4j_resolved_bjets_dPhi_llr  (new exists: SL_4j_resolved___bjets_dPhi_llr)
[DELETE] root://eoshome-a.cern.ch//eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_Rep/0805_LLR_odd/results/TTbar_sl_2022.root://SL_4j_resolved_bjets_dEta_llr  (new exists: SL_4j_resolved___bjets_dEta_llr)
[DELETE] root://eoshome-a.cern.ch//eos/user/a/anunezde/Z_OUTPUT_eos/Disc_Study_Rep/0805_LLR_odd